In [2]:
!pip install transformers torch tensorboardX
print("--- Installation Complete ---")

--- Installation Complete ---


In [3]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
import logging
import warnings
import random

logging.getLogger().setLevel(logging.CRITICAL)
warnings.filterwarnings('ignore')

device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")
print("--- Imports and Setup Complete ---")

Using CPU
--- Imports and Setup Complete ---


In [4]:
print("Loading pre-trained model and tokenizer (gpt2-medium)...")
MODEL_NAME = 'gpt2-medium'

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
model = model.to(device)

if tokenizer.pad_token is None:
    print("Tokenizer does not have a pad token, adding eos_token as pad_token.")
    tokenizer.add_special_tokens({'pad_token': tokenizer.eos_token})

    model.resize_token_embeddings(len(tokenizer))
    print(f"Resized model embeddings to: {len(tokenizer)}")


print("Model and tokenizer loaded.")
print(f"Tokenizer vocabulary size: {tokenizer.vocab_size}")
print(f"Model embedding size: {model.config.vocab_size}")
print("--- Model and Tokenizer Loading Complete ---")

Loading pre-trained model and tokenizer (gpt2-medium)...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Tokenizer does not have a pad token, adding eos_token as pad_token.
Resized model embeddings to: 50257
Model and tokenizer loaded.
Tokenizer vocabulary size: 50257
Model embedding size: 50257
--- Model and Tokenizer Loading Complete ---


In [5]:

CUSTOM_DATA_FILE = 'my_custom_data.txt'

if not os.path.exists(CUSTOM_DATA_FILE):
    print(f"Warning: '{CUSTOM_DATA_FILE}' not found. Creating a dummy file.")
    print(">>> Please upload your actual data file and restart the runtime or re-run this cell! <<<")
    with open(CUSTOM_DATA_FILE, 'w') as f:
        f.write("This is the first sentence of dummy data.\n")
        f.write("This is another sentence. You should replace this with your real data.\n")
        f.write("Upload your 'my_custom_data.txt' file to Colab for actual training.\n")

class CustomTextDataset(Dataset):
    def __init__(self, file_path, tokenizer):
        super().__init__()
        self.tokenizer = tokenizer
        self.text_list = []
        self.end_of_text_token = tokenizer.eos_token

        print(f"Reading data from {file_path}...")
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if line:

                        text_str = f"{line}{self.end_of_text_token}"
                        self.text_list.append(text_str)
            print(f"Loaded {len(self.text_list)} text items.")
            if not self.text_list:
                print("Warning: No text items loaded. Check your data file.")
        except FileNotFoundError:
            print(f"Error: Data file '{file_path}' not found.")

        except Exception as e:
            print(f"An error occurred while reading the file: {e}")


    def __len__(self):
        return len(self.text_list)

    def __getitem__(self, item):

        return self.text_list[item]

print("Creating Dataset and DataLoader...")
dataset = CustomTextDataset(CUSTOM_DATA_FILE, tokenizer)

dataloader = DataLoader(dataset, batch_size=1, shuffle=True)
print("Dataset and DataLoader ready.")
if len(dataset) == 0:
    print("!!! WARNING: Dataset is empty. Training cannot proceed. Check your data file and path. !!!")
print("--- Dataset Preparation Complete ---")

>>> Please upload your actual data file and restart the runtime or re-run this cell! <<<
Creating Dataset and DataLoader...
Reading data from my_custom_data.txt...
Loaded 3 text items.
Dataset and DataLoader ready.
--- Dataset Preparation Complete ---


In [6]:

GRAD_ACCUM_STEPS = 8
EPOCHS = 3
LEARNING_RATE = 3e-5
WARMUP_STEPS = 100
MAX_SEQ_LEN = model.config.n_positions

TRAIN_STEPS_PER_PRINT = 50


print(f"Effective Batch Size: {GRAD_ACCUM_STEPS}")
print(f"Epochs: {EPOCHS}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Warmup Steps: {WARMUP_STEPS}")
print(f"Max Sequence Length: {MAX_SEQ_LEN}")
print(f"Print Loss Every: {TRAIN_STEPS_PER_PRINT} gradient accumulation steps")

print("\nSetting up optimizer and scheduler...")


optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)


scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=-1)

print("Optimizer and scheduler ready.")
print("--- Hyperparameters and Optimizer Setup Complete ---")

Effective Batch Size: 8
Epochs: 3
Learning Rate: 3e-05
Warmup Steps: 100
Max Sequence Length: 1024
Print Loss Every: 50 gradient accumulation steps

Setting up optimizer and scheduler...
Optimizer and scheduler ready.
--- Hyperparameters and Optimizer Setup Complete ---


In [7]:
# Cell 6: Training Loop
print("Starting training...")
model.train()

proc_seq_count = 0
sum_loss = 0.0
batch_count = 0

tmp_text_tens = None

models_folder = "trained_models_custom"
if not os.path.exists(models_folder):
    os.makedirs(models_folder)
    print(f"Created folder: {models_folder}")

if len(dataset) == 0:
    print("!!! Cannot start training: Dataset is empty. Please check Cell 4. !!!")
else:
    for epoch in range(EPOCHS):
        print(f"\n======== EPOCH {epoch+1}/{EPOCHS} ========")
        tmp_text_tens = None
        proc_seq_count = 0

        for idx, text_item in enumerate(dataloader):

            text_str = text_item[0]


            text_tens = torch.tensor(tokenizer.encode(text_str)).unsqueeze(0).to(device)


            if text_tens.size()[1] > MAX_SEQ_LEN:
                print(f"Warning: Skipping item {idx+1} (length {text_tens.size()[1]} > MAX_SEQ_LEN={MAX_SEQ_LEN})")
                continue


            if not torch.is_tensor(tmp_text_tens):

                tmp_text_tens = text_tens
            else:

                if tmp_text_tens.size()[1] + text_tens.size()[1] > MAX_SEQ_LEN:

                    work_text_tens = tmp_text_tens


                    inputs = work_text_tens
                    labels = work_text_tens

                    outputs = model(inputs, labels=labels)
                    loss = outputs.loss


                    loss = loss / GRAD_ACCUM_STEPS
                    loss.backward()


                    sum_loss += outputs.loss.item()
                    proc_seq_count += 1


                    if proc_seq_count == GRAD_ACCUM_STEPS:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        optimizer.step()
                        scheduler.step()
                        optimizer.zero_grad()
                        model.zero_grad()
                        proc_seq_count = 0
                        batch_count += 1

                        if batch_count % TRAIN_STEPS_PER_PRINT == 0:
                            avg_loss = sum_loss / (TRAIN_STEPS_PER_PRINT * GRAD_ACCUM_STEPS)
                            print(f"  Epoch {epoch+1}/{EPOCHS} | Step {batch_count} | Avg Loss: {avg_loss:.4f}")
                            sum_loss = 0.0

                    tmp_text_tens = text_tens

                else:

                    tmp_text_tens = torch.cat([tmp_text_tens, text_tens], dim=1)



        print(f"\n--- Finished Epoch {epoch+1} ---")


        if torch.is_tensor(tmp_text_tens) and tmp_text_tens.size()[1] > 0:
             print(f"Processing remaining sequence buffer (length {tmp_text_tens.size()[1]})...")
             inputs = tmp_text_tens
             labels = tmp_text_tens
             outputs = model(inputs, labels=labels)
             loss = outputs.loss


             loss.backward()
             torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Optional clipping
             optimizer.step()
             scheduler.step()
             optimizer.zero_grad()
             model.zero_grad()
             print(f"Processed final buffer. Final Step Loss: {loss.item():.4f}")



        epoch_model_path = os.path.join(models_folder, f"{MODEL_NAME}_custom_epoch_{epoch+1}.pt")
        print(f"Saving model checkpoint for epoch {epoch+1} to {epoch_model_path}")
        try:

            torch.save(model.state_dict(), epoch_model_path)
            print("Model state_dict saved successfully.")
        except Exception as e:
            print(f"Error saving model: {e}")

    print("\n--- Training Finished ---")

Starting training...
Created folder: trained_models_custom

======== EPOCH 1/3 ========

--- Finished Epoch 1 ---
Processing remaining sequence buffer (length 45)...


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Processed final buffer. Final Step Loss: 4.2248
Saving model checkpoint for epoch 1 to trained_models_custom/gpt2-medium_custom_epoch_1.pt
Model state_dict saved successfully.

======== EPOCH 2/3 ========

--- Finished Epoch 2 ---
Processing remaining sequence buffer (length 45)...
Processed final buffer. Final Step Loss: 4.3731
Saving model checkpoint for epoch 2 to trained_models_custom/gpt2-medium_custom_epoch_2.pt
Model state_dict saved successfully.

======== EPOCH 3/3 ========

--- Finished Epoch 3 ---
Processing remaining sequence buffer (length 45)...
Processed final buffer. Final Step Loss: 4.5062
Saving model checkpoint for epoch 3 to trained_models_custom/gpt2-medium_custom_epoch_3.pt
Model state_dict saved successfully.

--- Training Finished ---


In [8]:

def choose_from_top(probs, n=5):
    """
    Selects a token ID randomly from the top N probabilities.
    Args:
        probs (np.array): Probabilities for each token in the vocabulary.
        n (int): The number of top probabilities to consider.
    Returns:
        int: The chosen token ID.
    """

    if not isinstance(probs, np.ndarray):
         probs = probs.cpu().numpy()


    ind = np.argpartition(probs, -n)[-n:]


    top_prob = probs[ind]

    top_prob = top_prob / np.sum(top_prob)


    choice = np.random.choice(n, 1, p=top_prob)
    token_id = ind[choice][0]

    return int(token_id)

print("--- Generation Helper Function Defined ---")

--- Generation Helper Function Defined ---


In [9]:

MODEL_LOAD_PATH = os.path.join(models_folder, f"{MODEL_NAME}_custom_epoch_{EPOCHS}.pt") # Load the last epoch's model
PROMPT = "The meaning of life is"  # Your starting prompt
MAX_GEN_LEN = 100
TOP_K = 20
TEMPERATURE = 1.0

# --- Load the Fine-Tuned Model State ---
print(f"Loading fine-tuned model state from: {MODEL_LOAD_PATH}")
if os.path.exists(MODEL_LOAD_PATH):
    try:

        model.load_state_dict(torch.load(MODEL_LOAD_PATH, map_location=device))
        model.eval()
        print("Model state loaded successfully.")
    except Exception as e:
        print(f"Error loading model state: {e}. Using the base pre-trained model for generation.")

else:
    print(f"Warning: Model file not found at {MODEL_LOAD_PATH}. Using the base pre-trained model (or last loaded state).")

    model.eval()

# --- Generate Text ---
print(f"\nGenerating text with prompt: '{PROMPT}'")
print("-" * 30)

prompt_encoded = tokenizer.encode(PROMPT, return_tensors='pt').to(device)
generated_sequence = prompt_encoded # Start the sequence with the prompt

with torch.no_grad():
    for _ in range(MAX_GEN_LEN):

        outputs = model(generated_sequence)

        next_token_logits = outputs.logits[:, -1, :]

        if TEMPERATURE > 0:
            next_token_logits = next_token_logits / TEMPERATURE


        next_token_probs = torch.softmax(next_token_logits, dim=-1)


        next_token_id = choose_from_top(next_token_probs.squeeze(), n=TOP_K)


        next_token_tensor = torch.tensor([[next_token_id]], dtype=torch.long, device=device)
        generated_sequence = torch.cat([generated_sequence, next_token_tensor], dim=1)


        if next_token_id == tokenizer.eos_token_id:
            print("\n<EOS token generated>")
            break

generated_text = tokenizer.decode(generated_sequence[0], skip_special_tokens=True)

print("\nGenerated Text:")
print(generated_text)
print("-" * 30)
print("--- Text Generation Complete ---")

Loading fine-tuned model state from: trained_models_custom/gpt2-medium_custom_epoch_3.pt
Model state loaded successfully.

Generating text with prompt: 'The meaning of life is'
------------------------------

Generated Text:
The meaning of life is not defined in stone. We are all connected to the universe through the Universe. The Universe has meaning. If it doesn't mean what we think it means, we have no meaning. But it has meaning to us. I will tell you one thing that is real to me. You know you have to take the truth of it. The universe has to mean something, if it hasn't meaning we have no meaning."

This was one more thing that was not true, and he was telling
------------------------------
--- Text Generation Complete ---


In [10]:

MODEL_EPOCH_TO_LOAD = EPOCHS
models_folder = "trained_models_custom"
model_path = os.path.join(models_folder, f"gpt2_medium_custom_epoch_{MODEL_EPOCH_TO_LOAD}.pt")


start_sentence = "بعد قراءة المقال، يمكن القول أن"

output_file_path = f'generated_text_epoch_{MODEL_EPOCH_TO_LOAD}.txt'
num_generations = 5
max_length = 150

print(f"Loading fine-tuned model from: {model_path}")
try:
    model.load_state_dict(torch.load(model_path, map_location=device))
    print("Model loaded successfully.")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please ensure the model file exists and training completed.")


model.eval()

print(f"\nGenerating {num_generations} text continuations...")
if os.path.exists(output_file_path):
    os.remove(output_file_path)

with torch.no_grad():
    for i in range(num_generations):
        print(f"\n--- Generation {i+1}/{num_generations} ---")
        print(f"Prompt: {start_sentence}")

        cur_ids = torch.tensor(tokenizer.encode(start_sentence)).unsqueeze(0).to(device)

        generated_sequence = []

        for j in range(max_length):
            outputs = model(cur_ids)
            logits = outputs.logits

            last_token_logits = logits[0, -1, :]
            softmax_logits = torch.softmax(last_token_logits, dim=0)


            if j < 5:
                n_sampling = 20
            else:
                n_sampling = 5
            next_token_id = choose_from_top(softmax_logits.to('cpu').numpy(), n=n_sampling)


            if next_token_id == tokenizer.eos_token_id:
                print("EOS token generated. Stopping generation.")
                break


            cur_ids = torch.cat([cur_ids, torch.tensor([[next_token_id]], device=device)], dim=1)
            generated_sequence.append(next_token_id)


        output_text = tokenizer.decode(generated_sequence)

        full_output = start_sentence + output_text

        print(f"Generated: {output_text}")


        with open(output_file_path, 'a', encoding='utf-8') as f:
            f.write(f"--- Generation {i+1} ---\n")
            f.write(f"Prompt: {start_sentence}\n")
            f.write(f"Generated Output:\n{full_output}\n\n")

print(f"\nFinished generation. Results saved to: {output_file_path}")

Loading fine-tuned model from: trained_models_custom/gpt2_medium_custom_epoch_3.pt
Error loading model: [Errno 2] No such file or directory: 'trained_models_custom/gpt2_medium_custom_epoch_3.pt'
Please ensure the model file exists and training completed.

Generating 5 text continuations...

--- Generation 1/5 ---
Prompt: بعد قراءة المقال، يمكن القول أن
Generated:  وفسى عن المسطر الأمواء على قبور من الشريق وشريق أمواء أن يعلى بأنصر الصلاة والحصاب القول بأنصر الفضيل يحرام عن عبد قبور الحصاب في الكريب كريب العباس

--- Generation 2/5 ---
Prompt: بعد قراءة المقال، يمكن القول أن
Generated:  العليث على يقتل بحديث المكن القول بالحمد عليها ، قوله قول إذا بى نصر على كرج ، يحج إذا كرج كحيط في الشافع إن تخبر الحمد عليها ، وأجله وأ

--- Generation 3/5 ---
Prompt: بعد قراءة المقال، يمكن القول أن
Generated:  المحجير والأصير والحسن عنه اعتجاء بالمقدم إن قدم يعشب بالصّر بعد المنزوق إلى الحسن عن تعضية في من يحجير بالعباسي بن من أصير ومقدم تجعل أن بقول الم

--- Generation 4/5 ---
Prompt: بعد قراءة المقال

KeyboardInterrupt: 